# SharkByte certified moving-branch verification — Colab CPU

This **pins the PR #22 merged source**, runs the audited signed-permutation mapping/replay harness, and exports a source-bound result. It does not allocate a dense high-qubit vector and does not claim general numerical certification. Colab RAM availability varies; the child process is limited to **at most 24 GiB virtual address space**, scaled down to detected headroom. Timeout or refusal is recorded, not counted as a pass.

Choose **CPU runtime**, then **Runtime → Run all**. No cloud API keys are needed.


In [ ]:
import hashlib, json, os, platform, resource, shutil, signal, subprocess, sys, tempfile, time, zipfile
from fractions import Fraction
from pathlib import Path

PINNED_SOURCE = "7793d5eace34456ce1cc57f85f4d75cf84c8e889"
RUN_DIR = Path(tempfile.mkdtemp(prefix="sharkbyte-moving-cloud-"))
REPO = RUN_DIR / "SharkByte"
OUT = RUN_DIR / "receipts"
OUT.mkdir()
if sys.platform != "linux":
    raise RuntimeError("Linux CPU runtime required")
subprocess.run(["git", "clone", "--quiet", "https://github.com/reggaesharkk/SharkByte.git", str(REPO)], check=True)
subprocess.run(["git", "checkout", "--detach", PINNED_SOURCE], cwd=REPO, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert actual == PINNED_SOURCE, (actual, PINNED_SOURCE)
print("Pinned source:", actual, "Environment:", platform.platform(), platform.machine(), sys.version.split()[0])


In [ ]:
# Memory is NOT guaranteed by Colab. Cap child address space at 24 GiB,
# or 65% of currently detected available memory, whichever is lower.
GIB = 1024**3
def available_bytes():
    mem = int(next(x.split()[1] for x in Path("/proc/meminfo").read_text().splitlines()
                   if x.startswith("MemAvailable:"))) * 1024
    for lim, used in (("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory.current"),
                      ("/sys/fs/cgroup/memory/memory.limit_in_bytes", "/sys/fs/cgroup/memory/memory.usage_in_bytes")):
        try:
            v = Path(lim).read_text().strip()
            if v.isdecimal():
                mem = min(mem, max(0, int(v) - int(Path(used).read_text().strip())))
        except (OSError, ValueError):
            pass
    return max(0, mem)
HEADROOM = available_bytes()
WORKER_AS_BYTES = min(24*GIB, int(HEADROOM * .65))
TIMEOUT_SECONDS = 1800
print(f"Available headroom: {HEADROOM/GIB:.2f} GiB; worker address-space ceiling: {WORKER_AS_BYTES/GIB:.2f} GiB")
PREFLIGHT_REFUSED = WORKER_AS_BYTES < GIB
if PREFLIGHT_REFUSED:
    print("REFUSED: insufficient memory headroom (<1 GiB child allowance)")


In [ ]:
if PREFLIGHT_REFUSED:
    summary = {"schema":"sharkbyte/colab-moving-cloud-manifest-v1",
               "source_commit":PINNED_SOURCE, "status":"REFUSED",
               "reason":"insufficient memory headroom (<1 GiB child allowance)",
               "machine":platform.machine(), "available_before_bytes":HEADROOM,
               "worker_address_space_limit_bytes":WORKER_AS_BYTES}
    (OUT/"manifest.json").write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n")
    print("Recorded result: REFUSED")
else:
    # Same audited harness used for the native Windows/WSL2 evidence.
    # No dense 2**qubits statevector is allocated.
    import tempfile
    command = [sys.executable, "-B", "tools/high_support/check_moving_branch_protection.py"]
    started = time.perf_counter()
    def child_limits():
        resource.setrlimit(resource.RLIMIT_AS, (WORKER_AS_BYTES, WORKER_AS_BYTES))
        os.setsid()
    with tempfile.TemporaryDirectory(prefix="moving-stage-", dir=RUN_DIR) as staging:
        raw = Path(staging) / "worker.json"
        error_log = Path(staging) / "stderr.log"
        with raw.open("wb") as stdout, error_log.open("wb") as stderr:
            proc = subprocess.Popen(command, cwd=REPO, stdout=stdout, stderr=stderr, preexec_fn=child_limits)
            try:
                code = proc.wait(timeout=TIMEOUT_SECONDS)
                outcome = "COMPLETED" if code == 0 else "REFUSED"
            except subprocess.TimeoutExpired:
                os.killpg(proc.pid, signal.SIGKILL)
                proc.wait()
                code, outcome = proc.returncode, "TIMEOUT"
        worker_log = error_log.read_text(errors="replace")[-10000:]
        summary = {"schema":"sharkbyte/colab-moving-cloud-manifest-v1",
                   "source_commit":PINNED_SOURCE, "status":outcome, "worker_exit":code,
                   "elapsed_seconds":time.perf_counter()-started,
                   "machine":platform.machine(), "host":platform.platform(), "python":sys.version.split()[0],
                   "available_before_bytes":HEADROOM, "worker_address_space_limit_bytes":WORKER_AS_BYTES}
        if outcome == "COMPLETED":
            try:
                record = json.loads(raw.read_text())
                assert record["implementation_commit"] == PINNED_SOURCE
                assert record["exhaustive_mappings"]["individual_gate_origin_mappings"] == 444
                assert record["exhaustive_composed_affine_mappings"][-1]["affine_coordinate_maps"] == 322560
                assert record["exhaustive_composed_affine_mappings"][-1]["basis_images_checked"] == 5160960
                for row in record["randomized_signed_permutation_circuits"]:
                    assert Fraction(row["exact_squared_error"]) <= Fraction(row["certified_l2_bound"])**2
                assert {v for v in record["decisions"].values() if v not in ("PASS", "OPEN", "UNSUPPORTED")} == set()
                for name, expected in record["source_hashes"].items():
                    assert hashlib.sha256((REPO/name).read_bytes()).hexdigest() == expected, name
                receipt = OUT / "moving-cloud.json"
                receipt.write_bytes(raw.read_bytes())
                summary.update(status="PASS", receipt_sha256=hashlib.sha256(receipt.read_bytes()).hexdigest(),
                               receipt_bytes=receipt.stat().st_size, checks=record["decisions"])
            except Exception as exc:
                summary.update(status="FAIL", reason=str(exc))
        if worker_log:
            (OUT/"worker-stderr.log").write_text(worker_log)
        (OUT/"manifest.json").write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n")
    print("Recorded result:",summary["status"],"elapsed s:",round(summary["elapsed_seconds"],2))
    print("Output:",OUT)


In [ ]:
# Export regardless of PASS/REFUSED/TIMEOUT; never relabel a refused run as PASS.
archive = RUN_DIR / "sharkbyte-moving-cloud-evidence.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUT.glob("*")):
        if path.is_file():
            bundle.write(path, arcname=path.name)
print("ZIP:", archive, "SHA-256:", hashlib.sha256(archive.read_bytes()).hexdigest())
from google.colab import files
files.download(str(archive))
